In [1]:
import polars as pl

lf = pl.scan_parquet("hf://datasets/calabi-yau-data/polytopes-4d/*.parquet")
row_count = lf.select(pl.len()).collect().item()

In [2]:
chunk_df = lf.slice(0, 100).collect()

In [25]:
import json

inputs = json.dumps({"verts": chunk_df[0, "vertices"].to_list()}, separators=(",", ":"))

!echo '{inputs}' | ../cone_nf/cone_nf

{"cones":[{"facet_verts":[[0,0,1,0],[1,0,0,0],[1,2,3,5],[0,1,0,0]],"cone_nf":[[1,0,0,0],[0,1,0,0],[0,0,1,0],[1,2,3,5]]},{"facet_verts":[[0,0,1,0],[1,0,0,0],[-2,-3,-4,-5],[0,1,0,0]],"cone_nf":[[1,0,0,0],[0,1,0,0],[0,0,1,0],[1,2,3,5]]},{"facet_verts":[[0,0,1,0],[1,0,0,0],[1,2,3,5],[-2,-3,-4,-5]],"cone_nf":[[1,0,0,0],[0,1,0,0],[0,0,1,0],[1,2,3,5]]},{"facet_verts":[[0,0,1,0],[1,2,3,5],[-2,-3,-4,-5],[0,1,0,0]],"cone_nf":[[1,0,0,0],[0,1,0,0],[0,0,1,0],[1,2,3,5]]},{"facet_verts":[[1,0,0,0],[1,2,3,5],[-2,-3,-4,-5],[0,1,0,0]],"cone_nf":[[1,0,0,0],[0,1,0,0],[0,0,1,0],[1,2,3,5]]}],"unique_cones":[{"cone_nf":[[1,0,0,0],[0,1,0,0],[0,0,1,0],[1,2,3,5]],"multiplicity":5,"example_facet":[[0,0,1,0],[1,0,0,0],[1,2,3,5],[0,1,0,0]]}]}


In [ ]:
import pyarrow.parquet as pq
import pyarrow.fs as pafs
import re

# ── Parameters ────────────────────────────────────────────────────────────────
S3_A_PREFIX = "s3://toriccy/datasets--calabi-yau-data--polytopes-4d/"    # "source" files (A)
S3_B_PREFIX = "s3://toriccy/shared/4d-polytope-facets//"   # "derived" files (B)
COLUMN_A    = "vertices"        # column to read from A; None → use row positions 0..N-1
COLUMN_B    = "vertices"     # column to read from B
PAIR_RE     = re.compile(r"polytopes-4d-(\d+-vertices).parquet$")  # stem used to pair files
B_PREFIX    = "4d-polytope-facets-"
B_SUFFIX    = ".parquet"   # A stem + B_SUFFIX → B filename
# ──────────────────────────────────────────────────────────────────────────────

s3 = pafs.S3FileSystem(region="us-east-2")

def to_hashable(v):
    if isinstance(v, list):
        return tuple(to_hashable(x) for x in v)
    return v

def list_parquets(s3_prefix):
    path = s3_prefix.removeprefix("s3://").rstrip("/")
    infos = s3.get_file_info(pafs.FileSelector(path, recursive=True))
    return {info.path for info in infos if info.path.endswith(".parquet")}

def read_key_set(s3_path, column):
    with s3.open_input_file(s3_path) as f:
        if column is None:
            return set(range(pq.ParquetFile(f).metadata.num_rows))
        vals = pq.read_table(f, columns=[column])[column].to_pylist()
        return {to_hashable(v) for v in vals}

a_files = list_parquets(S3_A_PREFIX)
b_files = list_parquets(S3_B_PREFIX)
b_root  = S3_B_PREFIX.removeprefix("s3://").rstrip("/")

results = []
for a_path in sorted(a_files):
    m = PAIR_RE.search(a_path)
    if not m:
        continue
    b_path  = f"{b_root}/{B_PREFIX}{m.group(1)}{B_SUFFIX}"
    a_keys  = read_key_set(a_path, COLUMN_A)
    b_keys  = read_key_set(b_path, COLUMN_B) if b_path in b_files else set()
    missing = sorted(a_keys - b_keys)
    if missing or b_path not in b_files:
        results.append((m.group(1), missing))

if results:
    total = sum(len(m) for _, m in results)
    print(f"{total} missing rows across {len(results)} file(s):\n")
    for stem, missing in results:
        preview = str(missing[:10]) + ("…" if len(missing) > 10 else "")
        print(f"  {stem}: {len(missing)} missing  {preview}")
else:
    print("All rows in A are present in B — no missing rows.")